# Lesson 2.4 — LLM-Enhanced PDF Understanding


**流程**：PyMuPDF 提取原始文本（快、免费）→ DeepSeek 做结构化理解（清洗 / 摘要 / 提取 / 转 JSON）

---

> **Vision OCR 说明**
>
> DeepSeek 当前 API 为**纯文本**，不支持图片。
>
> 真正的 Vision OCR 需要 GPT-4o / Claude / Gemini 等多模态模型。
>
> 本 demo 展示 **「文本提取 + LLM 后处理」** 的工程实践，对数字 PDF 效果相近。

交互式 UI 版本：`streamlit run 04_llm_ocr_demo.py`

## 知识扩展：LLM 时代的 PDF 解析

| 工具 | 原理 | 优势 |
|------|------|------|
| **LlamaParse** | LLM 端到端 | Markdown 输出、表格完美 |
| **Unstructured.io API** | 多模型组合 | 通用、稳定 |
| **GPT-4 Vision** | 直接看 PDF 截图 | 公式、图表都能理解 |
| **Marker** | DL 模型组合 | 开源、本地、Markdown |
| **Surya** | OCR + 版面 | 多语言、开源 |
| **Docling** (IBM) | 模型 + 规则 | 企业级，开源 |

### 趋势
- 端到端 LLM 解析（替代多工具流水线）
- Markdown 作为统一中间格式
- 多模态模型直接「看」PDF

**本 Notebook**：PyMuPDF 提文本 + DeepSeek 后处理（数字 PDF 低成本方案）。
扫描页见 [04_ocr_scanned.ipynb](04_ocr_scanned.ipynb) 或 Vision 模型。


## 0. 安装依赖 & 配置 API Key

In [ ]:
%pip install PyMuPDF openai python-dotenv -q

In [ ]:
import json
import os
from pathlib import Path

import fitz
from dotenv import load_dotenv
from openai import OpenAI

HERE = Path(".")
load_dotenv(HERE.parent / ".env")

DEEPSEEK_KEY = os.environ.get("DEEPSEEK_API_KEY", "")
MODEL = "deepseek-v4-flash"  # 可改为 deepseek-v4-pro
PDF_PATH = HERE / "sample.pdf"
PAGE_NUM = 1  # 处理第几页（1-based）

if not PDF_PATH.exists():
    raise FileNotFoundError(f"请把 PDF 放到: {PDF_PATH.resolve()}")

## 1. Step 1 — PyMuPDF 提取 & 页面预览

数字 PDF 直接用 PyMuPDF 提文本，无需 OCR。

In [ ]:
def render_page_image(pdf_path: str, page_idx: int, zoom: float = 2.0) -> bytes:
    doc = fitz.open(pdf_path)
    pix = doc[page_idx].get_pixmap(matrix=fitz.Matrix(zoom, zoom))
    doc.close()
    return pix.tobytes("png")


def extract_raw_text(pdf_path: str, page_idx: int) -> str:
    doc = fitz.open(pdf_path)
    text = doc[page_idx].get_text()
    doc.close()
    return text

In [ ]:
doc = fitz.open(PDF_PATH)
total_pages = len(doc)
doc.close()

page_idx = min(PAGE_NUM - 1, total_pages - 1)
raw_text = extract_raw_text(str(PDF_PATH), page_idx)

print(f"📄 {PDF_PATH.name}  共 {total_pages} 页，当前处理第 {page_idx + 1} 页")
print(f"字符数: {len(raw_text)}  |  Token 约 {len(raw_text) // 4}")
print("\n--- 原始文本 ---")
print(raw_text[:800])

In [ ]:
# 页面预览（需要 IPython）
try:
    from IPython.display import Image, display
    display(Image(render_page_image(str(PDF_PATH), page_idx)))
except ImportError:
    print("（安装 IPython 可显示页面预览图）")

## 2. Step 2 — 定义 LLM 任务

不同任务用不同的 system prompt + user prompt 模板。

In [ ]:
TASKS = {
    "清洗 & 格式化": {
        "system": "You are a document processing expert. Clean and format extracted PDF text.",
        "prompt": lambda t: (
            f"以下是从 PDF 提取的原始文本，可能含有排版噪声（换行错误、字符粘连等）。\n"
            f"请清洗并重新格式化，保留原意，输出结构清晰的 Markdown。\n\n"
            f"原始文本：\n{t}"
        ),
    },
    "文档摘要": {
        "system": "You are an expert at summarizing technical documents. Be concise and accurate.",
        "prompt": lambda t: f"请对以下文本生成一个简明摘要（3-5 句话），用中文回答：\n\n{t}",
    },
    "结构化提取（JSON）": {
        "system": "You are a structured data extraction expert. Always return valid JSON.",
        "prompt": lambda t: (
            f"从以下文档文本中提取关键信息，以 JSON 格式返回，包含：\n"
            f"- title: 文档标题\n"
            f"- key_topics: 主要话题列表\n"
            f"- entities: 重要实体（人名/模型名/组织等）\n"
            f"- numbers: 文中出现的重要数字指标\n\n"
            f"文本：\n{t}\n\n只返回 JSON，不要其他文字。"
        ),
    },
    "表格识别 → Markdown": {
        "system": "You are a table extraction expert. Identify and convert tables to Markdown.",
        "prompt": lambda t: (
            f"分析以下文本，找出所有表格数据（即使排版混乱），将其转换为标准 Markdown 表格格式。\n"
            f"如果没有表格，说明原因。\n\n文本：\n{t}"
        ),
    },
}

In [ ]:
def call_deepseek(prompt: str, system: str, model: str) -> str:
    client = OpenAI(api_key=DEEPSEEK_KEY, base_url="https://api.deepseek.com")
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content

## 3. Step 3 — 运行 LLM 处理

修改 `TASK_NAME` 切换任务。

In [ ]:
TASK_NAME = "文档摘要"  # 可选: 清洗 & 格式化 / 文档摘要 / 结构化提取（JSON） / 表格识别 → Markdown

if not DEEPSEEK_KEY:
    print("⚠️ 未找到 DEEPSEEK_API_KEY，请在 .env 中配置后再运行")
elif not raw_text.strip():
    print("⚠️ 当前页无文本（可能是扫描页，需要 Vision 模型或先做 OCR）")
else:
    cfg = TASKS[TASK_NAME]
    print(f"🤖 正在用 {MODEL} 执行: {TASK_NAME}...")
    result = call_deepseek(cfg["prompt"](raw_text), cfg["system"], MODEL)
    print("\n✅ LLM 输出：\n")
    print(result)

### 3.1 JSON 任务：解析结构化结果

In [ ]:
if DEEPSEEK_KEY and raw_text.strip() and TASK_NAME == "结构化提取（JSON）":
    try:
        clean = result.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        parsed = json.loads(clean)
        print("解析后的 JSON：")
        print(json.dumps(parsed, ensure_ascii=False, indent=2))
    except (NameError, json.JSONDecodeError) as e:
        print(f"JSON 解析失败: {e}")

## 4. 自定义 Prompt

In [ ]:
CUSTOM_PROMPT = "请分析以下文本并回答：这页主要讲了什么？\n\n{text}"

if DEEPSEEK_KEY and raw_text.strip():
    user_prompt = CUSTOM_PROMPT.replace("{text}", raw_text)
    custom_result = call_deepseek(
        user_prompt,
        "You are a helpful document analysis assistant.",
        MODEL,
    )
    print("自定义 Prompt 结果：\n")
    print(custom_result)

## 5. 原始 vs LLM 处理后对比

In [ ]:
# 对比视图（需先运行 Step 3）
try:
    print("=" * 40 + " 原始提取 " + "=" * 40)
    print(raw_text[:1500])
    print("\n" + "=" * 40 + " LLM 处理后 " + "=" * 40)
    print(result[:1500])
except NameError:
    print("请先运行 Step 3 生成 result")

## 6. 小结

| 方案 | 适用 | 成本 |
|------|------|------|
| PyMuPDF + LLM 后处理 | 数字 PDF | 低（仅 LLM token） |
| Vision OCR (GPT-4o 等) | 扫描 PDF / 复杂版面 | 高（图片 token） |
| Tesseract OCR + LLM | 扫描 PDF 批量处理 | 中（本地 OCR 免费） |

**工程建议**：先判 PDF 类型 → 数字页走本课方案 → 扫描页走 [04_ocr_scanned.ipynb](04_ocr_scanned.ipynb)。